# ResRank stage 1 (0.6B) on a Kaggle T4

**Before running:** Settings -> Accelerator: *GPU T4 x2* (one GPU is used) and Internet: *On*. Add as inputs the code dataset and
`resrank-checkpoints`. Add-ons -> Secrets: attach `KAGGLE_API_TOKEN` (use a NEW token).

Run the cells in order. **If the session dies, start a new one and run the same cells again: training resumes by itself.**
GPU hours count while a session is on, so stop the session when you are done.

In [ ]:
# 1. Token and budget (the token is read from a Secret and never printed)
import os
from kaggle_secrets import UserSecretsClient
os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
HOURS_LEFT = 27.0      # GPU hours left this week
print("token loaded:", bool(os.environ.get("KAGGLE_API_TOKEN")))

In [ ]:
%%bash
# 2. Setup: copy the code, install, restore earlier checkpoints
rm -rf /kaggle/working/resrank
cp -r /kaggle/input/datasets/tanmaysarda1/it550-final-project/resrank /kaggle/working/resrank
cd /kaggle/working/resrank
pip install -q -e .
pip install -q transformers huggingface_hub pyyaml pytest bitsandbytes kagglehub
mkdir -p /kaggle/working/checkpoints
CK=/kaggle/input/datasets/tanmaysarda1/resrank-checkpoints
# restore ONLY stage1_full (every checkpoint copy costs 4.8 GB of the 19.5 GB working disk)
if [ -d "$CK/stage1_full" ]; then cp -r "$CK/stage1_full" /kaggle/working/checkpoints/; echo "restored stage1_full:"; ls /kaggle/working/checkpoints/stage1_full; else echo "no stage1_full in the dataset (fine on a first run)"; fi
python -c "import resrank; print('resrank imported from', resrank.__file__)"
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
# 3. Unit tests (about 40 seconds): every test should pass (about 150)
cd /kaggle/working/resrank
python -m pytest -q 2>&1 | tail -3

In [ ]:
%%bash
# 4. Download the PE-Rank training data (skipped by the downloader if it is already there)
cd /kaggle/working/resrank
python -m resrank.data.s01_download pe_rank
ls -la /kaggle/working/data/01_raw/pe_rank

In [ ]:
%%bash
# 5. Timing run (about 5 minutes): measures real speed so the sample count can be planned. Skipped if a plan is saved.
cd /kaggle/working/resrank
if [ -f /kaggle/working/checkpoints/stage1_full/plan.json ]; then echo "saved plan found: skipping the timing run"; exit 0; fi
python -m resrank.data.s02_build_train stage1 --input /kaggle/working/data/01_raw/pe_rank/train.jsonl --name timing --n 160 --n-candidates 16
python -u -m resrank.train.run_training --train-config configs/stage1_small.yaml --train-file timing --out-name timing_run --no-publish --no-save
python -m resrank.train.log_summary /kaggle/working/checkpoints/timing_run

In [ ]:
# 6. Plan the number of training samples from the measured speed and the hours left.
#    The plan is saved next to the checkpoints, so every later session (resume) uses the SAME number.
import json, pathlib, sys
sys.path.insert(0, "/kaggle/working/resrank/src")
from resrank.train.log_summary import summarize
from resrank.train.plan import plan_samples

N_OVERRIDE = None      # e.g. 30000 to choose your own; ignored if a saved plan already exists
plan_path = pathlib.Path("/kaggle/working/checkpoints/stage1_full/plan.json")
if plan_path.exists():
    plan = json.loads(plan_path.read_text())
    print("reusing the saved plan (it must not change, or --resume will refuse to continue):")
else:
    s = summarize("/kaggle/working/checkpoints/timing_run")
    print("timing run:", s)
    if s["oom_skipped_steps"]:
        print("WARNING: the timing run skipped steps because of out-of-memory. Stop and check before continuing.")
    plan = plan_samples(s["sec_per_sample_incl_warmup"], HOURS_LEFT)
    if N_OVERRIDE:
        plan.update(n_samples=N_OVERRIDE, optimizer_steps=N_OVERRIDE // 16, note="manual override")
    plan_path.parent.mkdir(parents=True, exist_ok=True)
    plan_path.write_text(json.dumps(plan))
print(json.dumps(plan, indent=2))
os.environ["N_SAMPLES"] = str(plan["n_samples"])

In [ ]:
%%bash
# 7. Build the stage-1 training file with the planned sample count (deterministic: the same on every session)
cd /kaggle/working/resrank
python -m resrank.data.s02_build_train stage1 --input /kaggle/working/data/01_raw/pe_rank/train.jsonl --n $N_SAMPLES --n-candidates 16
cat /kaggle/working/data/02_train/stage1.stats.json
echo "--- disk (the working folder is limited to about 19.5 GB; one checkpoint is 4.8 GB) ---"
df -h /kaggle/working | tail -1
du -sh /kaggle/working/data/* /kaggle/working/checkpoints/* 2>/dev/null

In [ ]:
%%bash
# 8. TRAIN. Shows a progress bar. Starts fresh, resumes from the newest snapshot, or does nothing if finished.
#    A snapshot is saved and uploaded to the Kaggle dataset every 15 minutes.
cd /kaggle/working/resrank
python -u -m resrank.train.run_training --train-config configs/stage1_small.yaml --auto-resume

In [ ]:
%%bash
# 9. Summary of the training log: loss at the start vs the end, skipped steps, speed
cd /kaggle/working/resrank
python -m resrank.train.log_summary /kaggle/working/checkpoints/stage1_full

## Evaluation on SciFact (run after training; needs the GPU, about 15 to 25 minutes)
Compares BM25, the untrained models, the earlier 2,000-sample run (if restored) and the full run.

In [ ]:
%%bash
# 10. Download SciFact and make BM25 candidates for its test queries (the download URL is the standard BEIR one, unverified)
cd /kaggle/working/resrank
python -m resrank.data.s01_download_beir --name scifact
D=/kaggle/working/data/01_raw/beir/scifact
python -m resrank.data.s03_bm25 --dataset scifact --corpus $D/corpus.jsonl --queries $D/queries.jsonl --qrels $D/qrels/test.tsv

In [ ]:
%%bash
# 11. Encode, rerank and score: BM25 vs untrained vs trained (nDCG@10).
#     While training is unfinished the weights are in stage1_full/latest, so that is used when there is no final checkpoint.
cd /kaggle/working/resrank
D=/kaggle/working/data/01_raw/beir/scifact
R=/kaggle/working/data
ARGS="--run bm25=$R/03_candidates/scifact.bm25.run"
for CK in pretrained stage1_full; do
  SRC=$CK
  if [ "$CK" != "pretrained" ]; then
    if [ -d /kaggle/working/checkpoints/$CK/encoder ]; then SRC=$CK
    elif [ -d /kaggle/working/checkpoints/$CK/latest/encoder ]; then SRC=$CK/latest; echo "using the periodic snapshot $SRC"; cat /kaggle/working/checkpoints/$SRC/train_state.json; echo
    else echo "skipping $CK (not present)"; continue; fi
  fi
  TAG=$(echo $SRC | tr '/' '_')
  if [ ! -f $R/05_runs/scifact.$TAG.original.run ]; then
    python -m resrank.eval.s06_encode_corpus --dataset scifact --corpus $D/corpus.jsonl --ckpt $SRC --only-candidates $R/03_candidates/scifact.bm25.run
    python -m resrank.eval.s07_rerank --dataset scifact --queries $D/queries.jsonl --ckpt $SRC --tag $TAG --orders original
  else echo "run for $TAG already exists: reusing it"; fi
  ARGS="$ARGS --run $TAG=$R/05_runs/scifact.$TAG.original.run"
done
python -m resrank.eval.s08_evaluate --dataset scifact --qrels $D/qrels/test.tsv $ARGS | grep "{'dataset'"

In [ ]:
%%bash
# 12. Manual upload of the finished checkpoints (training already uploads at the end; use this if that failed)
cd /kaggle/working/resrank
python -m resrank.train.publish --dataset tanmaysarda1/resrank-checkpoints --names stage1_full

## Using both GPUs: train in the background on GPU 0, evaluate snapshots on GPU 1
Use these instead of cell 8 (do not run both). The evaluation cell takes a hard-linked copy of the newest snapshot, so it
costs no extra disk and stays consistent even if training replaces `latest` while it runs.

In [ ]:
%%bash
# 8b. Start training in the background on GPU 0 (starts fresh, resumes, or does nothing if finished)
cd /kaggle/working/resrank
CUDA_VISIBLE_DEVICES=0 setsid nohup python -u -m resrank.train.run_training --train-config configs/stage1_small.yaml --auto-resume > /kaggle/working/train.log 2>&1 < /dev/null &
sleep 3
echo "training started in the background on GPU 0; log: /kaggle/working/train.log"
pgrep -af run_training | cut -c1-90

In [ ]:
%%bash
# 8c. Monitor: progress bar line, processes, GPU memory, newest snapshot, disk. Run it whenever you like.
echo "--- progress ---"
tail -c 800 /kaggle/working/train.log | tr '\r' '\n' | grep -E "training:|WARNING|publish|finished|Error|Traceback|already" | tail -4
echo "--- processes ---"; pgrep -af run_training | cut -c1-90
echo "--- GPUs ---"; nvidia-smi --query-gpu=index,memory.used,memory.total,utilization.gpu --format=csv,noheader
echo "--- newest snapshot ---"; cat /kaggle/working/checkpoints/stage1_full/latest/train_state.json 2>/dev/null; echo
df -h /kaggle/working | tail -1

In [ ]:
%%bash
# 8d. Evaluate the newest snapshot on GPU 1 while training continues on GPU 0 (about 10 to 15 minutes)
cd /kaggle/working/resrank
D=/kaggle/working/data/01_raw/beir/scifact
R=/kaggle/working/data
CKD=/kaggle/working/checkpoints
STEP=$(python -c "import json; print(json.load(open('$CKD/stage1_full/latest/train_state.json'))['step'])")
SNAP=snap_step$STEP
rm -rf $CKD/$SNAP
cp -al $CKD/stage1_full/latest $CKD/$SNAP      # hard links: no extra disk, and a stable copy if training replaces 'latest'
echo "evaluating the snapshot from step $STEP"
CUDA_VISIBLE_DEVICES=1 python -m resrank.eval.s06_encode_corpus --dataset scifact --corpus $D/corpus.jsonl --ckpt $SNAP --only-candidates $R/03_candidates/scifact.bm25.run
CUDA_VISIBLE_DEVICES=1 python -m resrank.eval.s07_rerank --dataset scifact --queries $D/queries.jsonl --ckpt $SNAP --orders original
rm -rf $CKD/$SNAP
ARGS="--run bm25=$R/03_candidates/scifact.bm25.run --run step$STEP=$R/05_runs/scifact.$SNAP.original.run"
[ -f $R/05_runs/scifact.pretrained.original.run ] && ARGS="$ARGS --run pretrained=$R/05_runs/scifact.pretrained.original.run"
python -m resrank.eval.s08_evaluate --dataset scifact --qrels $D/qrels/test.tsv $ARGS | grep "{'dataset'"

In [ ]:
# 8e. Is a difference between two runs real? Paired bootstrap over the 300 test queries (95% interval).
#     Edit the two run files: A and B can be any two runs, e.g. a snapshot vs BM25, or a later snapshot vs an earlier one.
import sys
sys.path.insert(0, "/kaggle/working/resrank/src")
from resrank.io_utils import read_trec_run, read_qrels
from resrank.eval.metrics import evaluate_run, paired_bootstrap

R = "/kaggle/working/data"
qrels = read_qrels(f"{R}/01_raw/beir/scifact/qrels/test.tsv")
RUN_A = f"{R}/05_runs/scifact.stage1_full_latest.original.run"     # e.g. the snapshot
RUN_B = f"{R}/03_candidates/scifact.bm25.run"                      # e.g. BM25
def per_query(path):
    return evaluate_run(read_trec_run(path), qrels, "ndcg", 10)["per_query"]
a, b = per_query(RUN_A), per_query(RUN_B)
qs = sorted(qrels)
res = paired_bootstrap([a[q] for q in qs], [b[q] for q in qs], n_boot=10000, seed=0)
print({k: round(v, 4) for k, v in res.items()})
print("A is better with certainty" if res["ci_low"] > 0 else "B is better with certainty" if res["ci_high"] < 0
      else "the 95% interval includes 0: the difference is not clearly real")

In [ ]:
%%bash
# 8f. Stop the background training (the newest snapshot stays; rerun 8b to resume from it)
pkill -f run_training; sleep 5; pgrep -af run_training || echo "stopped"